# IBR Dynamic Modeling and Simulation

Nine teaching lessons. The solver uses only Python's standard library; ibrsim is not required. The low-frequency experiments use nominal-frequency algebraic networks with ideal current/voltage realizations, separate from the full-order source models. Keep parameters, ports, residuals and scope with every run. The website provides interactive plots.


In [ ]:
"""IBR course teaching models; Python standard library only.

Adapted conventions/control laws from PINN-IBR, reviewed 2026-10-03.
These are independently authored teaching realizations, not ibrsim models.
Network: balanced nominal-frequency algebraic impedances. GFL: ideal current
actuator + SRF PLL (4 states); droop: ideal voltage source + PQ filters (3);
VSM: adds rotor speed (4). Parallel: both branches share one PCC (7).
Switch: explicit GFL -> VSM reset, with voltage and frequency matching.
LCL: separate open-loop averaged six-state electrical plant in seconds.
No switching ripple, DC dynamics, saturation, faults, or protection.
"""
import math
import cmath

WB = 2 * math.pi * 50
NAMES = {"gfl": ["delta", "xi_pll", "id", "iq"],
         "droop": ["delta", "Pf", "Qf"],
         "vsm": ["delta", "omega", "Pf", "Qf"]}


def bounded(value, low, high, name):
    value = float(value)
    if not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f"{name} must be finite and in [{low}, {high}]")
    return value


def settings(case):
    mode = case.get("mode", "droop")
    if mode not in ("frame", "lcl", "gfl", "droop", "vsm", "parallel", "switch", "compare"):
        raise ValueError("Unknown model mode")
    event = case.get("event", "p")
    if event not in ("p", "q", "v", "f"):
        raise ValueError("Unknown disturbance")
    limit = .01 if mode == "lcl" else .5 if event == "f" else .08
    return dict(mode=mode, event=event,
                step=bounded(case.get("step", .005 if mode == "lcl" else .03), -limit, limit, "step"),
                scr=bounded(case.get("scr", 5), 2, 10, "SCR"),
                mp=bounded(case.get("mp", .02), .005, .04, "m_p"),
                inertia=bounded(case.get("inertia", 4), .5, 8, "M"),
                bandwidth=bounded(case.get("bandwidth", 2), .5, 5, "PLL bandwidth"),
                angle=bounded(case.get("angle", 20), -180, 180, "frame angle"),
                dt=bounded(case.get("dt", .0005), .0001, .001, "time step"),
                duration=bounded(case.get("duration", 4), 4, 20, "duration"))


def rk4(rhs, x, h):
    a = rhs(x)
    b = rhs([v + h * d / 2 for v, d in zip(x, a)])
    c = rhs([v + h * d / 2 for v, d in zip(x, b)])
    d = rhs([v + h * e for v, e in zip(x, c)])
    return [v + h * (aa + 2 * bb + 2 * cc + dd) / 6
            for v, aa, bb, cc, dd in zip(x, a, b, c, d)]


def high_voltage_power_flow(s, z):
    """V = 1 + Z conj(S/V), choosing the high-voltage root."""
    a, b = (z * s.conjugate()).real, (z * s.conjugate()).imag
    discriminant = 1 + 4 * a - 4 * b * b
    if discriminant <= 0:
        raise ValueError("No qualified high-voltage equilibrium")
    magnitude_squared = (1 + 2 * a + math.sqrt(discriminant)) / 2
    return complex(magnitude_squared - a, b)


class TeachingCase:
    def __init__(self, kind, cfg):
        self.cfg = cfg
        self.kinds = ["gfl", "droop"] if kind == "parallel" else ["gfl" if kind == "switch" else kind]
        self.zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
        self.zf = complex(.00625, .1)
        self.e0, self.commands, self.slices, self.x0 = [], [], [], []
        voltage = high_voltage_power_flow(complex(.6, 0), self.zg)
        current = complex(.6 / len(self.kinds), 0).conjugate() / voltage.conjugate()
        for family in self.kinds:
            start = len(self.x0)
            if family == "gfl":
                local = current * cmath.exp(-1j * cmath.phase(voltage))
                state = [cmath.phase(voltage), 0, local.real, local.imag]
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), 0.])
            else:
                source = voltage + self.zf * current
                state = [cmath.phase(source), .6 / len(self.kinds), 0]
                if family == "vsm":
                    state.insert(1, 1.)
                self.e0.append(1.)
                self.commands.append([.6 / len(self.kinds), (abs(source) - 1) / .0325])
            self.x0.extend(state)
            self.slices.append((start, len(self.x0)))

    def evaluate(self, x, after=False):
        cfg = self.cfg
        step = cfg["step"] if after else 0
        voltage_grid = 1 + (step if cfg["event"] == "v" else 0)
        omega_grid = 1 + (step / 50 if cfg["event"] == "f" else 0)
        dp = step / len(self.kinds) if cfg["event"] == "p" else 0
        dq = step / len(self.kinds) if cfg["event"] == "q" else 0
        forcing, denominator, entries = complex(voltage_grid), complex(1), []
        for j, (family, (start, stop)) in enumerate(zip(self.kinds, self.slices)):
            xb = x[start:stop]
            if family == "gfl":
                value = complex(xb[2], xb[3]) * cmath.exp(1j * xb[0])
                forcing += self.zg * value
            else:
                qf = xb[3] if family == "vsm" else xb[2]
                amplitude = self.e0[j] - .0325 * (qf - self.commands[j][1] - dq)
                if not .5 < amplitude < 1.5:
                    raise ValueError("Voltage source left the teaching model domain")
                value = amplitude * cmath.exp(1j * xb[0])
                forcing += self.zg * value / self.zf
                denominator += self.zg / self.zf
            entries.append(value)
        voltage = forcing / denominator
        if not .5 < abs(voltage) < 1.5:
            raise ValueError("PCC voltage left the teaching model domain")
        derivatives, currents, powers, omegas = [], [], [], []
        for j, (family, (start, stop), value) in enumerate(zip(self.kinds, self.slices, entries)):
            xb = x[start:stop]
            current = value if family == "gfl" else (value - voltage) / self.zf
            s = voltage * current.conjugate()
            p_ref, q_ref = self.commands[j][0] + dp, self.commands[j][1] + dq
            if family == "gfl":
                local = voltage * cmath.exp(-1j * xb[0])
                error = local.imag / abs(voltage)
                wn = 2 * math.pi * cfg["bandwidth"]
                omega = 1 + 2 * .707 * wn / WB * error + wn * wn / WB * xb[1]
                ref = complex(p_ref, -q_ref) / local.conjugate()
                derivative = [WB * (omega - omega_grid), error,
                              (ref.real - xb[2]) / .02, (ref.imag - xb[3]) / .02]
            elif family == "droop":
                omega = 1 - cfg["mp"] * (xb[1] - p_ref)
                derivative = [WB * (omega - omega_grid), (s.real - xb[1]) / .1, (s.imag - xb[2]) / .05]
            else:
                omega = xb[1]
                derivative = [WB * (omega - omega_grid),
                              (p_ref - xb[2] - (omega - 1) / cfg["mp"]) / cfg["inertia"],
                              (s.real - xb[2]) / .1, (s.imag - xb[3]) / .05]
            derivatives.extend(derivative)
            currents.append(current)
            powers.append(s)
            omegas.append(omega)
        current = sum(currents)
        total = voltage * current.conjugate()
        residual = abs(voltage - voltage_grid - self.zg * current)
        return dict(dx=derivatives, voltage=voltage, currents=currents, powers=powers,
                    p=total.real, q=total.imag, v=abs(voltage), frequency=50 * omegas[0],
                    residual=residual, omega=omegas[0])

    def reset_to_vsm(self, x):
        before = self.evaluate(x, True)
        source = before["voltage"] + self.zf * before["currents"][0]
        s, omega = before["powers"][0], before["omega"]
        dp = self.cfg["step"] if self.cfg["event"] == "p" else 0
        dq = self.cfg["step"] if self.cfg["event"] == "q" else 0
        self.commands[0][0] = s.real + (omega - 1) / self.cfg["mp"] - dp
        self.e0[0] = abs(source) + .0325 * (s.imag - self.commands[0][1] - dq)
        self.kinds[0] = "vsm"
        reset = [cmath.phase(source), omega, s.real, s.imag]
        after = self.evaluate(reset, True)
        error = max(abs(after["voltage"] - before["voltage"]), abs(after["p"] - before["p"]),
                    abs(after["q"] - before["q"]), abs(after["omega"] - omega))
        return reset, error


def controller_run(kind, cfg):
    model = TeachingCase(kind, cfg)
    x = model.x0[:]
    initial_residual = max(map(abs, model.evaluate(x)["dx"]))
    time, traces = [], {key: [] for key in ("p", "q", "v", "frequency")}
    if kind == "parallel":
        traces.update(p_gfl=[], p_gfm=[])
    network_residual, reset_error = 0., None
    # Events land exactly on step boundaries; each segment has frozen inputs.
    segments = [(0., 1., False), (1., 2. if kind == "switch" else cfg["duration"], True)]
    if kind == "switch":
        segments.append((2., cfg["duration"], True))
    for segment, (start, stop, after) in enumerate(segments):
        if kind == "switch" and segment == 2:
            x, reset_error = model.reset_to_vsm(x)
        count = math.ceil((stop - start) / cfg["dt"])
        h = (stop - start) / count
        stride = max(1, round(.01 / h))
        for k in range(count + 1):
            if k % stride == 0 or k == count:
                e = model.evaluate(x, after)
                network_residual = max(network_residual, e["residual"])
                time.append(stop if k == count else start + k * h)
                for key in ("p", "q", "v", "frequency"):
                    traces[key].append(e[key])
                if kind == "parallel":
                    traces["p_gfl"].append(e["powers"][0].real)
                    traces["p_gfm"].append(e["powers"][1].real)
            if k < count:
                x = rk4(lambda state: model.evaluate(state, after)["dx"], x, h)
    manifest = [f"{family}.{name}" for family in model.kinds for name in NAMES[family]]
    return dict(time=time, traces=traces, mode=kind, state_count=len(x), state_names=manifest,
                initial_residual=initial_residual, network_residual=network_residual,
                reset_error=reset_error, final_commands=model.commands,
                scope="nominal-frequency algebraic network; ideal current/voltage realization")


def frame_run(cfg):
    offset = cfg["angle"] * math.pi / 180
    time = [k / 10000 for k in range(401)]
    traces = {key: [] for key in ("va", "vb", "vc", "vd", "vq", "p", "q")}
    residual = 0.
    for t in time:
        phases = [WB * t + shift for shift in (0, -2 * math.pi / 3, 2 * math.pi / 3)]
        va = [math.sqrt(2 / 3) * math.cos(p) for p in phases]
        ia = [math.sqrt(2 / 3) * .6 * math.cos(p - math.pi / 6) for p in phases]
        vd = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, va))
        vq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, va))
        id_ = sum(math.sqrt(2 / 3) * math.cos(p + offset) * a for p, a in zip(phases, ia))
        iq = sum(-math.sqrt(2 / 3) * math.sin(p + offset) * a for p, a in zip(phases, ia))
        p, q = vd * id_ + vq * iq, vq * id_ - vd * iq
        residual = max(residual, abs(p - sum(v * i for v, i in zip(va, ia))))
        for key, value in zip(traces, [*va, vd, vq, p, q]):
            traces[key].append(value)
    return dict(time=time, traces=traces, mode="frame", state_count=0, initial_residual=residual,
                network_residual=0., reset_error=None, scope="balanced power-invariant Park transform")


def lcl_run(cfg):
    r1, ell1, cf = .00625, 8.4375e-5, .0008
    zg = complex(1, 10) / (cfg["scr"] * math.sqrt(101))
    r2, ell2 = .00625 + zg.real, (.1 + zg.imag) / WB
    i2 = complex(.6)
    vc = 1 + complex(r2, WB * ell2) * i2
    i1 = i2 + 1j * WB * cf * vc
    u = vc + complex(r1, WB * ell1) * i1
    x = [i1.real, i1.imag, vc.real, vc.imag, i2.real, i2.imag]
    def evaluate(state, after):
        a, v, b = complex(*state[:2]), complex(*state[2:4]), complex(*state[4:])
        command = u + (cfg["step"] if after else 0)
        da = (command - v - r1 * a) / ell1 - 1j * WB * a
        dv = (a - b) / cf - 1j * WB * v
        db = (v - 1 - r2 * b) / ell2 - 1j * WB * b
        denergy = ell1 * (a.conjugate() * da).real + cf * (v.conjugate() * dv).real + ell2 * (b.conjugate() * db).real
        balance = (command * a.conjugate()).real - b.real - r1 * abs(a)**2 - r2 * abs(b)**2
        return [da.real, da.imag, dv.real, dv.imag, db.real, db.imag], abs(denergy - balance)
    initial = max(map(abs, evaluate(x, False)[0]))
    time, traces, residual = [], {key: [] for key in ("i2d", "vcd", "vcq")}, 0.
    for start, stop, after in [(0., .01, False), (.01, .03, True)]:
        count = round((stop - start) / .000002)
        h = (stop - start) / count
        for k in range(count + 1):
            if k % 25 == 0 or k == count:
                time.append(stop if k == count else start + k * h)
                for key, value in zip(traces, [x[4], x[2], x[3]]):
                    traces[key].append(value)
                residual = max(residual, evaluate(x, after)[1])
            if k < count:
                x = rk4(lambda state: evaluate(state, after)[0], x, h)
    return dict(time=time, traces=traces, mode="lcl", state_count=6, initial_residual=initial,
                network_residual=residual, reset_error=None, scope="open-loop six-state averaged LCL; energy balance audit")


def solve(case):
    cfg = settings(case)
    if cfg["mode"] == "frame":
        result = frame_run(cfg)
    elif cfg["mode"] == "lcl":
        result = lcl_run(cfg)
    elif cfg["mode"] == "compare":
        runs = {kind: controller_run(kind, cfg) for kind in ("gfl", "droop", "vsm", "parallel")}
        first = runs["gfl"]
        result = dict(time=first["time"], traces={kind: run["traces"]["p"] for kind, run in runs.items()},
                      mode="compare", state_count=0, initial_residual=max(r["initial_residual"] for r in runs.values()),
                      network_residual=max(r["network_residual"] for r in runs.values()), reset_error=None,
                      runs={kind: {k: v for k, v in run.items() if k not in ("time", "traces")} for kind, run in runs.items()},
                      scope=first["scope"])
    else:
        result = controller_run(cfg["mode"], cfg)
    result["case"] = cfg
    return result



## C1-01 · System Boundaries and Model Representations

<h3>Start with the question and the terminal</h3>
<p>An inverter-based resource connects a controlled converter to a network. A model must tell us where its boundary is: the average converter voltage, the filter capacitor, the point of connection (PoC), or a plant-level bus. A power measurement at the capacitor differs from a PoC measurement because the intervening branch stores energy and dissipates power.</p>
<p>For this course, draw <strong>converter → L₁ → capacitor C → L₂ → PoC → grid impedance → infinite bus</strong>. The infinite bus fixes voltage magnitude and frequency. The DC supply is ideal. Positive P and Q denote injection into the network at the declared measurement port. Controls receive reference commands and measurements; they do not directly prescribe every electrical state.</p>
<h3>Three representations of the same equipment</h3>
<div class="ibr-table"><table><thead><tr><th>Representation</th><th>Retains</th><th>Useful question</th></tr></thead><tbody><tr><td>Switching model</td><td>Individual switching actions and their electrical consequences</td><td>What ripple does a switching pattern produce?</td></tr><tr><td>Averaged dq model</td><td>Average voltage command, plant and controller dynamics</td><td>How do current control and LCL dynamics interact?</td></tr><tr><td>Positive-sequence / low-frequency model</td><td>Selected envelope, synchronization and power dynamics</td><td>How does a small command change affect synchronization?</td></tr></tbody></table></div>
<p>Representation and controller family are separate choices. A GFL controller can be represented with either an explicit LCL plant or an ideal current actuator. A low-order GFM voltage source can hide inner voltage and current loops. Removing those loops changes the questions the model can answer.</p>
<h3>Write an interface before an equation</h3>
<p>Use x for differential states, z for algebraic network variables, u for commands and grid conditions, and y for measured outputs. The network constraint must be solved consistently with the controller. A state is a stored quantity with an evolution law; a current obtained by solving an impedance relation is an algebraic variable.</p>
<p><strong>Worked classification:</strong> in the course droop experiment, x = [δ, P_f, Q_f]. The PCC voltage and branch current belong to z, while P*, Q*, V_g and ω_g are inputs. The capacitor voltage is absent from x: this experiment cannot display capacitor resonance. Module 3 adds it explicitly.</p>
<h3>What the first experiment establishes</h3>
<p>All four traces below start at total PoC P = 0.6 pu and Q = 0. An increase of 0.03 pu at 1 s produces different trajectories because the retained control dynamics differ. Similar endpoints do not prove that two models are interchangeable. Keep the state list, measurement port, parameter set and disturbance with every result.</p>

$$
\dot{x}=f(x,z,u),\qquad 0=g(x,z,u),\qquad y=h(x,z,u)
$$

1. Run the default comparison and identify which dynamics are retained in each trace.
2. Choose a voltage step and explain why these traces cannot establish switching-ripple accuracy.
3. Write a four-line model contract: boundary, states, inputs, measured outputs.


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


In the three-state droop experiment, which quantity is algebraic?

1. Filtered active power
2. PCC voltage
3. Internal angle

<details><summary>Explanation</summary>

PCC voltage is obtained from the network constraint. P_f and δ have differential equations. In an explicit capacitor model, that capacitor voltage becomes a state; classification depends on the chosen representation.

</details>


## C1-02 · Reference Frames and Per-Unit Conventions

<h3>Rotate coordinates without changing physical power</h3>
<p>For a balanced three-phase signal with no zero sequence, the power-invariant Park matrix maps abc into two rotating axes. Its rows use cos(θ + φₖ) and −sin(θ + φₖ), with φₖ = [0, −2π/3, 2π/3]. The √(2/3) scaling preserves the three-phase inner product. Do not combine this transform with the 3/2 power coefficient from an amplitude-invariant convention.</p>
<p>The inverse on the balanced subspace is the transpose of the Park matrix. A zero-sequence component requires an additional axis; the two-axis inverse cannot recover it.</p>
<h3>Power and current references</h3>
<p>With current positive toward the network, P = v_d i_d + v_q i_q and Q = v_q i_d − v_d i_q. When the local d axis aligns with voltage, v_q = 0: positive Q requires negative i_q. The current-reference matrix below is the exact inverse of this P–Q map for nonzero voltage.</p>
<h3>Per unit still has units behind it</h3>
<p>For S_b = 10 kVA, V_b = 400 V line-to-line RMS and f_b = 50 Hz, Z_b = 16 Ω and ω_b = 314.159 rad/s. The power-invariant dq voltage base is 400 V and dq current base is 25 A. The physical line RMS current base is 14.434 A; it is a different quantity.</p>
<p>Define r = R/Z_b, ℓ = L/Z_b in seconds, and c = C Z_b in seconds. Thus inductive reactance at nominal frequency is x = ω_b ℓ. A per-unit inductance written as reactance x cannot be substituted for ℓ in a time-domain derivative.</p>
<h3>A numerical check</h3>
<p>The experiment uses voltage magnitude 1, current magnitude 0.6 and a 30° current lag. At zero frame offset, v_d = 1, v_q = 0, P = 0.6 cos(30°) = 0.519615 and Q = 0.3. At +20° offset, v_d = cos(20°), v_q = −sin(20°). P and Q stay unchanged because voltage and current rotate together.</p>
<p>An angle relative to the grid obeys δ̇ = ω_b(ω_pu − ω_g,pu). A difference of 0.001 pu at 50 Hz gives 0.314159 rad/s, not 0.001 rad/s.</p>

$$
T(\theta)=\sqrt{\frac{2}{3}}\begin{bmatrix}\cos\theta&\cos(\theta-2\pi/3)&\cos(\theta+2\pi/3)\\-\sin\theta&-\sin(\theta-2\pi/3)&-\sin(\theta+2\pi/3)\end{bmatrix}
$$

$$
P=v_d i_d+v_q i_q,\qquad Q=v_q i_d-v_d i_q
$$

$$
\begin{bmatrix}i_d^*\\i_q^*\end{bmatrix}=\frac{1}{v_d^2+v_q^2}\begin{bmatrix}v_d&v_q\\v_q&-v_d\end{bmatrix}\begin{bmatrix}P^*\\Q^*\end{bmatrix}
$$

$$
Z_b=\frac{V_b^2}{S_b},\quad I_{b,dq}=\frac{S_b}{V_b},\quad \dot\delta=\omega_b(\omega_{pu}-\omega_{g,pu})
$$

1. Set the frame offset to 0°, +20° and −90°; compare v_d and v_q.
2. Select the P–Q plot and verify that the physical power is invariant.
3. Edit the current lag in frame_run() and predict the signs of P and Q before running.


In [ ]:
case = {"mode": "frame", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


At v_d = 1 and v_q = 0, which current injects Q = +0.2 pu?

1. i_q = +0.2 pu
2. i_q = 0.3 pu
3. i_q = −0.2 pu

<details><summary>Explanation</summary>

Q = −v_d i_q in this aligned frame, so i_q = −0.2. The answer follows from the stated transform and injection convention; changing either requires updating the full set of equations.

</details>


## C1-03 · Averaged Converter and LCL Plant

<h3>Retain the six electrical states</h3>
<p>An averaged converter imposes a continuous voltage command u rather than individual switching events. The LCL plant retains converter-side current i₁, capacitor voltage v_c and output current i₂, each with d and q components. The grid-side filter and grid inductance carry the same current, so combine their series r and ℓ for the differential equation; reconstruct the PoC voltage when a separate terminal measurement is needed.</p>
<p>Use J = [[0, 1], [−1, 0]]. In a frame rotating at ω_b ω, a complex dq vector acquires the term −jω_bω times itself. The real form is +ω_bωJ times the vector. These signs follow directly from differentiating a rotating coordinate transform.</p>
<h3>Commands and closed-loop states are different</h3>
<p>The experiment holds the converter voltage at its equilibrium value, then adds 0.005 pu to its d component at 10 ms. It is an <strong>open-loop plant experiment</strong>. A closed-loop droop GFM also needs angle, P–Q filters, voltage PI and current PI states. Six electrical states plus one angle, two power filters and four PI integrators give the repository’s 13-state droop realization.</p>
<h3>Initialize all components consistently</h3>
<p>Take i₂ = 0.6 − j0 at the infinite-bus port. Compute v_c = 1 + (r₂ + jω_bℓ₂)i₂, i₁ = i₂ + jω_b c v_c and u = v_c + (r₁ + jω_bℓ₁)i₁. Starting the capacitor at 1 and both currents at zero would introduce an unintended energization transient.</p>
<p>The default coefficients are r₁ = 0.00625, ℓ₁ = 84.375 μs, c = 0.0008 s, r₂ = 0.00625 + r_g and ω_bℓ₂ = 0.1 + x_g. The open-loop case uses a 2 μs integration step over 30 ms. These are declared teaching parameters, not a reproduction of every source notebook parameter profile.</p>
<h3>Audit energy, not only the plotted curve</h3>
<p>Define W = (ℓ₁|i₁|² + c|v_c|² + ℓ₂|i₂|²)/2. Cross-axis terms cancel in Ẇ. The energy-rate identity below checks voltage signs, branch directions and loss terms together. Its residual is an equation-consistency check, not an external validation of the converter.</p>

$$
\dot i_1=\frac{u-v_c-r_1i_1}{\ell_1}+\omega_b\omega Ji_1
$$

$$
\dot v_c=\frac{i_1-i_2}{c}+\omega_b\omega Jv_c,\qquad\dot i_2=\frac{v_c-v_g-r_2i_2}{\ell_2}+\omega_b\omega Ji_2
$$

$$
v_{PoC}=v_c-r_{f2}i_2-\ell_{f2}(\dot i_2-\omega_b\omega Ji_2)
$$

$$
\dot W=u^Ti_1-v_g^Ti_2-r_1\lVert i_1\rVert^2-r_2\lVert i_2\rVert^2
$$

1. Run the +0.005 pu converter-voltage step and inspect the first 20 ms after the event.
2. Reduce SCR from 5 to 2 and rerun; explain what changed in the electrical branch.
3. Inspect lcl_run(): identify each stored-energy term and verify the energy-rate residual.


In [ ]:
case = {"mode": "lcl", "scr": 5, "step": 0.005, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Which states does this open-loop LCL experiment contain?

1. Two currents and one voltage, each with d and q components
2. Six controller integrators
3. Only active and reactive power

<details><summary>Explanation</summary>

The six states are i₁d, i₁q, v_cd, v_cq, i₂d and i₂q. Power is computed from electrical variables; it becomes a separate state only when an explicit measurement filter is added.

</details>


## C1-04 · PLL and Grid-Following Control

<h3>Follow the voltage angle, then control current</h3>
<p>A grid-following controller estimates the network angle with a phase-locked loop and expresses voltage and current in that estimated frame. The SRF PLL drives local v_q toward zero. Positive v_q means the voltage vector leads the estimated d axis; the stated PLL sign increases the estimated angle.</p>
<p>Our four-state realization uses δ, the PLL integral ξ, and two ideal current-actuator states i_d and i_q. A 20 ms current-actuator time constant replaces the detailed current controller and LCL plant. It is useful for studying synchronization and command tracking, but cannot evaluate inner-loop resonance or current-limit behavior.</p>
<h3>Normalize the PLL and preserve the time scale</h3>
<p>The phase detector is e = v_q/|V|. Its PI output is a per-unit frequency offset: ω̂ = 1 + k_p e + k_i ξ, with ξ̇ = e. The relative angle obeys δ̇ = ω_b(ω̂ − ω_g). Near alignment with a stiff voltage, the characteristic polynomial is s² + ω_b k_p s + ω_b k_i.</p>
<p>Set k_p = 2ζω_n/ω_b and k_i = ω_n²/ω_b, where ζ = 0.707 and ω_n = 2πb. The lab’s “PLL bandwidth” b is a natural-frequency design parameter in Hz; it is not a measured closed-loop −3 dB bandwidth in every network.</p>
<h3>Map power to current at the measured port</h3>
<p>Use the full inverse P–Q map from Module 2, rather than dividing both commands by v_d while assuming v_q = 0 throughout a transient. The local current rotates into the grid frame, then the network gives V = V_g + Z_g I. This feedback makes voltage angle and PLL response depend on grid impedance.</p>
<p>The baseline holds total PoC P = 0.6 and Q = 0. A P-command step changes current demand; a grid-frequency step makes the PLL track the new frequency. This teaching GFL has no frequency–watt law, volt–var law or RoCoF injection. Those are separate controls in the source’s higher-order GFL models.</p>
<h3>Interpret a weak-grid experiment carefully</h3>
<p>Decrease SCR while holding the commands and gains fixed. Compare P, Q, PCC magnitude and estimated frequency. A smooth response in this model establishes only that this ideal-actuator realization remained bounded for the declared test. The repository’s full 15-state GFL also retains measurement filters and physical plant states.</p><h3>Reconnect the detailed current-control plant</h3><p>For the source LCL GFL, a PoC P–Q command first produces grid-side current reference i₂*. With capacitor-current compensation enabled, i₁* = i₂* − ω_b ω̂ c Jv_c. Define e_i = i₁* − i₁ and integrate ξ̇_i = e_i. The current PI, voltage feedforward and inductive decoupling generate the average converter voltage u; then the six LCL equations determine actual current. The full 15-state realization retains one relative angle, one PLL integral, two voltage-measurement states, one frequency-measurement state, two power-command filter states, two current PI integrals and six electrical states. The four-state browser model is a distinct ideal-actuator realization.</p>

$$
e=\frac{v_q}{\lVert v\rVert},\quad\dot\xi=e,\quad\hat\omega=1+k_pe+k_i\xi,\quad\dot\delta=\omega_b(\hat\omega-\omega_g)
$$

$$
k_p=\frac{2\zeta\omega_n}{\omega_b},\qquad k_i=\frac{\omega_n^2}{\omega_b}
$$

$$
\tau_i\dot i_{dq}=i_{dq}^*-i_{dq},\qquad V=V_g+Z_gI
$$

$$
i_1^*=i_2^*-\omega_b\hat\omega cJv_c,\quad\dot\xi_i=i_1^*-i_1
$$

$$
u=v_c-\omega_b\hat\omega\ell_1Ji_1+K_{pi}(i_1^*-i_1)+K_{ii}\xi_i
$$

1. Run the baseline P step, then select estimated frequency and PCC magnitude.
2. Apply a +0.1 Hz grid-frequency step; check the PLL frequency after the transient.
3. At SCR = 2, compare PLL design settings of 1 and 5 Hz using the same disturbance.


In [ ]:
case = {"mode": "gfl", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Local v_q is positive. Which initial PLL action is consistent with this convention?

1. Decrease the estimated angular speed
2. Increase the estimated angular speed
3. Set P* to zero automatically

<details><summary>Explanation</summary>

With positive k_p, positive v_q gives positive frequency correction, advancing the estimated d axis toward the voltage vector. This PLL equation alone does not change the power command.

</details>


## C1-05 · Droop Grid-Forming Control

<h3>Create an angle from a power–frequency law</h3>
<p>A droop GFM creates an internal voltage angle by integrating its frequency command. It does not need a PLL for this synchronization law. A filtered active-power mismatch changes frequency; a filtered reactive-power mismatch changes voltage magnitude. The network then determines actual injected power.</p>
<p>Our three states are δ, P_f and Q_f. The voltage realization is E exp(jδ) behind Z_f = 0.00625 + j0.1 pu. The branch current is (E exp(jδ) − V)/Z_f. The physical LCL and PI loops are replaced by this algebraic relation. Z_f is a declared source impedance in the teaching model.</p>
<h3>The loop closes through the network</h3>
<p>For a small positive P* step, the frequency initially rises because P_f cannot jump. δ advances, electrical power increases, and P_f catches up. At the restored equilibrium against a nominal-frequency infinite bus, ω = 1 and P_f = P*. The reference is a control input, not a direct assignment to the measured power trajectory.</p>
<p>Increasing m_p changes both the steady frequency–power slope and the transient loop gain. The fixed measurement filters are τ_p = 0.1 s and τ_q = 0.05 s. Do not attribute every response change solely to “more droop” without also considering the network and filter time scales.</p>
<h3>Match a terminal operating point</h3>
<p>The experiment initializes all families at PoC P = 0.6 and Q = 0. Its internal source amplitude generally exceeds 1 because the source impedance carries current. With E₀ fixed at 1, Q* must be chosen so that E = E₀ − n_q(Q_f − Q*) equals that required amplitude. Therefore the droop Q* is not zero even when terminal Q is zero.</p>
<p>At SCR = 5, the baseline Q* is approximately 0.316795 pu. That is an equilibrium-matching control offset, not a claim that the PCC exports that reactive power. The lab exposes the actual commands so this distinction can be inspected.</p>
<h3>A frequency step tests the slope</h3>
<p>Apply +0.1 Hz to the grid. At a locked steady state, ω = ω_g and P ≈ P* − (ω_g − 1)/m_p. For m_p = 0.02, the predicted active-power change is −0.1/(50 × 0.02) = −0.1 pu. This is a within-model steady-state prediction; the algebraic impedances remain evaluated at nominal frequency.</p><h3>Build the full cascaded-control realization</h3><p>To retain inner loops, use x = [δ, P_f, Q_f, ξ_vd, ξ_vq, ξ_id, ξ_iq, i₁d, i₁q, v_cd, v_cq, i₂d, i₂q]. Set v* = [E, 0], voltage error e_v = v* − v_c and ξ̇_v = e_v. With output-current feedforward and capacitor decoupling enabled, the voltage PI produces i₁* using the first added equation below. Current error is e_i = i₁* − i₁ and ξ̇_i = e_i; the current PI produces converter voltage u using the second equation. Insert u into the LCL equations from Module 3 to close the model.</p><p>The canonical source droop realization feeds its power filters from v_c and i₂; its benchmark outputs are reconstructed at PoC. Those two ports must remain distinct. To initialize a no-limit equilibrium, first determine the electrical state, then set ξ_v = (i₁ − F i₂ + ω_b ω c Jv_c)/K_iv and ξ_i = (u − v_c + ω_b ω ℓ₁ Ji₁)/K_ii. Zero controller error does not imply zero PI integrals. The browser droop lab deliberately removes these four integrators and six electrical states.</p>

$$
\tau_p\dot P_f=P_{PoC}-P_f,\qquad\tau_q\dot Q_f=Q_{PoC}-Q_f
$$

$$
\omega=1-m_p(P_f-P^*),\quad E=E_0-n_q(Q_f-Q^*),\quad\dot\delta=\omega_b(\omega-\omega_g)
$$

$$
I=\frac{Ee^{j\delta}-V}{Z_f},\qquad P+jQ=VI^*
$$

$$
e_v=v^*-v_c,\quad\dot\xi_v=e_v,\quad i_1^*=F i_2-\omega_b\omega cJv_c+K_{pv}e_v+K_{iv}\xi_v
$$

$$
e_i=i_1^*-i_1,\quad\dot\xi_i=e_i,\quad u=v_c-\omega_b\omega\ell_1Ji_1+K_{pi}e_i+K_{ii}\xi_i
$$

1. Run +0.03 pu P*, then inspect frequency: explain why it changes before power settles.
2. Inspect the matched Q* and compare it with measured Q at time zero.
3. Apply +0.1 Hz and compare the final P change with −0.1/(50 m_p).


In [ ]:
case = {"mode": "droop", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


When terminal Q = 0, must this droop model also have Q* = 0?

1. Yes, for every network impedance
2. Only if the PLL gain is zero
3. No; Q* can provide the internal voltage offset required by the branch

<details><summary>Explanation</summary>

At equilibrium E = E₀ + n_q Q* when Q_f = 0. Current through Z_f requires a particular E, so Q* matches that source amplitude. A terminal target and a controller command are different quantities.

</details>


## C1-06 · VSM and Virtual Inertia

<h3>Add a frequency state, rather than a name</h3>
<p>A virtual synchronous machine replaces algebraic P–frequency droop with an evolving internal speed. The four-state teaching realization contains δ, ω, P_f and Q_f. Its electrical source and reactive-power law are the same as Module 5. This isolates the effect of the synchronization law.</p>
<p>Use Mω̇ = P* − P_f − D(ω − 1), with M in seconds and D = 1/m_p. If M = 2H under this per-unit swing convention, the default M = 4 s corresponds to H = 2 s. The angle equation still multiplies per-unit frequency difference by ω_b.</p>
<h3>Keep steady slope separate from inertia</h3>
<p>At steady state, P_f = P* − D(ω_g − 1), the same slope as the droop controller when D = 1/m_p. M affects the transient speed derivative, not this steady slope. The source repository separates governor and explicit damping coefficients, constraining their sum to 1/m_p; our lab uses the aggregate coefficient.</p>
<p>Immediately after a +0.03 pu P* step, at the old equilibrium, ω̇ = 0.03/4 = 0.0075 pu/s. Expressed in Hz/s that is 0.375. Doubling M halves this initial derivative when all other initial conditions and coefficients are held fixed.</p>
<h3>What supports the electrical power?</h3>
<p>“Virtual inertia” describes a control law. The lab assumes an ideal DC source and unlimited voltage/current realization. It cannot establish DC energy availability or converter overload capability. To study those questions, add DC storage dynamics, device limits and their control interactions.</p>
<h3>Do not merge GFL virtual inertia with VSM</h3>
<p>A GFL virtual-inertia controller can add a causal filtered RoCoF-dependent power request while retaining PLL-based synchronization. That is a different model structure from an internal VSM speed state. The repository’s GFL-VI notebook includes an unstable frozen-parameter example; adding an inertia-labelled term is not a stability guarantee.</p>
<p>Compare M = 1, 4 and 8 s with the same SCR, droop slope and measurement filters. Report initial RoCoF, peak power and settling behavior. A reduced VSM response cannot certify the stability of the repository’s higher-order electrical and inner-control realizations.</p><p>The default 4 s window may end before VSM settles, especially with the retained 0.1 s power filter. Extend the duration to 20 s before discussing a steady value. Some parameter choices can produce growing oscillations; preserve that result and check the model domain.</p>

$$
M\dot\omega=P^*-P_f-D(\omega-1),\qquad D=\frac1{m_p},\qquad M=2H
$$

$$
\dot\delta=\omega_b(\omega-\omega_g),\qquad P_{\infty}=P^*-\frac{\omega_g-1}{m_p}
$$

$$
\dot f(1^+)=f_b\frac{\Delta P^*}{M}
$$

1. With M = 4 s, predict the initial frequency derivative for a +0.03 pu step.
2. Repeat at M = 1 and 8 s while holding m_p and SCR fixed.
3. Use the frequency-step experiment to compare the steady slope with droop GFM.


In [ ]:
case = {"mode": "vsm", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Doubling M with fixed D and operating point does what immediately after the same P* step?

1. Halves the initial frequency derivative
2. Doubles the steady frequency–power slope
3. Removes the need for a DC energy source

<details><summary>Explanation</summary>

Initially Mω̇ = ΔP*, so ω̇ is inversely proportional to M. At equilibrium ω̇ = 0 and M disappears from the steady power–frequency relation.

</details>


## C1-07 · Parallel GFL–GFM Hybrid Models

<h3>Couple branches through a shared terminal</h3>
<p>The lab connects one GFL current-actuator branch and one droop-GFM voltage-source branch to a single PCC. They are simultaneously active. Each receives half of the total P-command increment. Their outputs are coupled through the same grid impedance; combining two separately simulated single-inverter traces would miss this interaction.</p>
<p>The total system base remains 10 kVA. Both branches use that common base, each starts at P = 0.3 pu and Q = 0, and the total starts at 0.6 pu. When using different individual device bases, convert currents, powers and impedances before applying the shared network equations.</p>
<h3>Solve the network once</h3>
<p>For grid impedance Z_g, GFL current I_c and GFM source U = E exp(jδ) behind Z_f, KCL gives I_grid = I_c + (U − V)/Z_f. Combined with V = V_g + Z_g I_grid, this produces the explicit PCC expression below. This equation must be solved before evaluating either branch’s power feedback.</p>
<p>The state count is 4 + 3 = 7. State concatenation alone is insufficient: both controllers must see the same physical PCC voltage, expressed in their own local frames. Compute each branch power as V Iₖ* and total power as V(I_c + I_v)*.</p>
<h3>Branch power is not a weighted output guess</h3>
<p>The default command step is split equally, but the two branch transients need not be equal because their controller dynamics differ. The “branch power” plot shows the GFL and GFM contributions together with the total. Their instantaneous sum must equal total PCC power.</p>
<h3>Relation to REGFM_C1</h3>
<p>This is a <strong>shared-PCC parallel teaching model</strong>, not the official REGFM_C1 implementation. The source notebook distinguishes its positive-sequence A11 formulation from a detailed-average F21 research lift. A virtual R + jX relation and a physical LCL circuit have different retained dynamics. Use the source notebook when reproducing those exact controller equations and parameter definitions.</p>
<p>Decrease SCR and inspect both branch powers and total power. Any observed interaction is specific to this current-source/voltage-source realization, common-base convention and power-sharing policy. It is not a general ranking of hybrid controller architectures.</p>

$$
I_g=I_c+\frac{U-V}{Z_f},\qquad V=V_g+Z_gI_g
$$

$$
V=\frac{V_g+Z_g I_c+(Z_g/Z_f)U}{1+Z_g/Z_f}
$$

$$
S_{PCC}=V(I_c+I_v)^*=S_c+S_v
$$

1. Select branch power and compare the two contributions during the +0.03 pu total step.
2. Check that the branch powers sum to the total at the event and at the final time.
3. Reduce SCR to 2, rerun, and describe which branch response changes more.


In [ ]:
case = {"mode": "parallel", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Why is adding independently simulated branch traces insufficient?

1. Powers cannot be added at a common terminal
2. It omits the common-PCC voltage feedback and network coupling
3. Parallel branches must have identical states

<details><summary>Explanation</summary>

Powers do add when evaluated at the same PCC. The missing part in independent simulations is the shared voltage: each branch changes it and therefore changes the other branch’s feedback.

</details>


## C1-08 · Mode-Switching Hybrid Models

<h3>A mode label changes which equations are active</h3>
<p>A mode-switching inverter uses one active controller at a time. Let m ∈ {GFL, VSM}, with ẋ = f_m(x,u) between events and x⁺ = R(x⁻,u) at a switching event. An active-state vector and a union of saved checkpoint variables are different objects; the union is not automatically a larger ODE.</p>
<p>This demonstration starts in the four-state GFL realization, applies a command disturbance at 1 s, and switches to the four-state VSM realization at 2 s. Although both vectors have length four, their meanings differ: [δ_PLL, ξ_PLL, i_d, i_q] cannot be copied into [δ_source, ω, P_f, Q_f].</p>
<h3>Construct a terminal-compatible reset</h3>
<p>Before switching, record PCC V, injected I, P, Q and PLL ω. The incoming source must satisfy U⁺ = V⁻ + Z_f I⁻. Set its angle to arg(U⁺), its frequency to ω⁻, and its power filters to P⁻ and Q⁻. These conditions reconstruct the same terminal voltage and current through the incoming source impedance.</p>
<p>Set E₀⁺ = |U⁺| + n_q(Q⁻ − Q*) and P*⁺ = P⁻ + D(ω⁻ − 1). The voltage offset is matched, and initial VSM acceleration is zero. This is an explicit policy that <strong>recalibrates the incoming command and voltage offset</strong>. It does not promise that the pre-switch P* remains unchanged.</p>
<h3>Measure what is continuous</h3>
<p>The reset audit measures the largest jump in complex PCC voltage, P, Q and internal frequency. The default should be close to floating-point roundoff. The source angle can change because the incoming source sits behind an impedance while the outgoing angle belongs to the PLL. Angle equality would be the wrong continuity criterion here.</p>
<p>Algebraic voltage can still jump at the separate disturbance event at 1 s. The reset conditions apply at 2 s and should not be confused with general continuity under every input change.</p>
<h3>What remains for a detailed transition model</h3>
<p>The lab has one scheduled transition and no inactive-controller tracking, limiter, dwell timer or protection guard. The source VSM–PLL transition notebook additionally treats hold/release policies and different active-state dimensions. It also separates a stable controller-specific demonstration profile from an unstable same-gain GFM endpoint. Endpoint tests alone do not validate a same-device mode transition.</p>

$$
\dot x=f_m(x,u),\qquad x^+=R_{m^-\to m^+}(x^-,u)
$$

$$
U^+=V^-+Z_f I^-,\quad\delta^+=\arg U^+,\quad\omega^+=\omega^-,\quad P_f^+=P^-,\quad Q_f^+=Q^-
$$

$$
E_0^+=|U^+|+n_q(Q^--Q^*),\qquad P^{*+}=P^-+D(\omega^--1)
$$

1. Run the scheduled switch and inspect the trace immediately before and after 2 s.
2. Read the reset residual and the final command offsets in the numerical audit.
3. In reset_to_vsm(), deliberately replace the reset angle by the PLL angle; inspect the resulting jump.


In [ ]:
case = {"mode": "switch", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


Can equal-length GFL and VSM state vectors be copied directly at a switch?

1. Yes, equal length guarantees compatibility
2. Yes, if both use per unit
3. No; reset must map physical meanings and terminal conditions

<details><summary>Explanation</summary>

The GFL integral and currents are not VSM speed and filtered powers. Reconstruct the incoming source from V and I, then map frequency and power measurements. State semantics, rather than vector length, determine the reset.

</details>


## C1-09 · Equilibrium, Disturbances, and Fair Comparison

<h3>Choose targets at the same physical port</h3>
<p>A fair comparison starts from common PoC P, Q, voltage, frequency, system base and grid conditions. Controller commands may differ to realize that target. In the lab, total P = 0.6 pu and Q = 0 for GFL, droop, VSM and parallel cases. The parallel case divides the terminal target into two equal branch contributions.</p>
<p>Use S_b = 10 kVA, V_b = 400 V and f_b = 50 Hz. Grid magnitude is initially 1 pu, X/R = 10 and |Z_g| = 1/SCR. These assumptions define the teaching short-circuit ratio on a common system base. Parallel branch ratings are not used to silently change the grid impedance.</p>
<h3>Solve equilibrium before applying a disturbance</h3>
<p>At the terminal, S = VI* and V = 1 + Z_g I. Eliminating I gives V = 1 + Z_g conj(S/V). Select the high-voltage root, initialize each controller in its own coordinates, and verify max|f(x₀,u₀)|. Droop and VSM Q* offsets compensate the source-impedance voltage drop; forcing identical raw Q* would produce different terminal operating points.</p>
<h3>Keep the event protocol reproducible</h3>
<p>Integrate 0–1 s with the old input, then 1–4 s with the new input. The solver places events exactly on integration boundaries; adjacent samples at 1 s show the before and after values. This prevents the Runge–Kutta stages from averaging across a discontinuity. Switch experiments additionally end a segment at 2 s and apply an explicit reset.</p>
<p>The default time step is 0.5 ms. Re-run with half that value in the Python experiment and compare trajectories, rather than treating a smooth plot as evidence of adequate numerical accuracy.</p>
<h3>Separate three kinds of evidence</h3>
<div class="ibr-table"><table><thead><tr><th>Evidence</th><th>Check here</th><th>What it establishes</th></tr></thead><tbody><tr><td>Equation consistency</td><td>Equilibrium, KCL, power sums and reset residuals</td><td>The stated equations and interface agree</td></tr><tr><td>Model validity</td><td>Compare against an explicit higher-order realization over a declared range</td><td>A bounded approximation error for that study</td></tr><tr><td>External validation</td><td>Independent measurements or a qualified reference model</td><td>Agreement with that external evidence</td></tr></tbody></table></div>
<p>The website experiments establish the first row. The source six-family comparison notebook retains stable and unstable rows explicitly and documents its common operating-point protocol. Do not omit an unstable model merely to produce attractive overlays, or infer a global controller ranking from one step test.</p>
<h3>Course project</h3>
<p>Submit a model contract, your parameter snapshot and Python experiment, a matched-equilibrium table, two disturbance comparisons, and a short explanation of observed differences. Include numerical refinement, retained outputs and validity limits. An extended project can repeat the protocol using the source full-order notebooks and report how the teaching-model conclusions change.</p><p>The default 4 s window may end before VSM settles, especially with the retained 0.1 s power filter. Extend the duration to 20 s before discussing a steady value. Some parameter choices can produce growing oscillations; preserve that result and check the model domain.</p>

$$
S=VI^*,\qquad V=1+Z_g\overline{S/V},\qquad\lVert f(x_0,u_0)\rVert_\infty<\varepsilon
$$

$$
Z_g=\frac{1+j\,10}{\mathrm{SCR}\sqrt{101}},\qquad\Delta P^*_{branch}=\frac{\Delta P^*_{total}}{N}
$$

$$
e_h=\max_k|P_h(t_k)-P_{h/2}(t_k)|
$$

1. Compare all four models at SCR = 5 with the +0.03 pu total P step.
2. Repeat with a +0.1 Hz frequency step; explain the GFL versus GFM steady responses.
3. In Python, halve case["dt"] and compare aligned samples; retain the maximum error in the report.


In [ ]:
case = {"mode": "compare", "scr": 5, "step": 0.03, "event": "p", "duration": 4}
result = solve(case)
print("states:", result["state_count"])
print("initial residual:", result["initial_residual"])
print("network / energy residual:", result["network_residual"])
print("reset:", result["reset_error"])
print("final:", {k: values[-1] for k, values in result["traces"].items()})


What should be matched before comparing different controller families?

1. The terminal operating point, base and disturbance protocol
2. Every raw controller command
3. Only the number of states

<details><summary>Explanation</summary>

Different control laws require different internal commands to produce the same terminal P and Q. Compare declared common terminal conditions and preserve model-specific state initialization. Equal raw commands alone do not make the comparison fair.

</details>
